<a href="https://colab.research.google.com/github/Narendra725/Power_BI_Spark_Labs/blob/main/Power%20BI/Automations/Power%20Bi%20Desktop/Mach3/power_bi_objects_creation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install datamodel-code-generator

In [ ]:
import requests
import json
import re
import os
from datetime import datetime, timedelta, timezone
from urllib.parse import urljoin
from datamodel_code_generator import InputFileType, generate

# Define Official Fabric Schema URLs
urls = {
    "Report": "https://developer.microsoft.com/json-schemas/fabric/item/report/definition/report/3.0.0/schema.json",
    "Page": "https://developer.microsoft.com/json-schemas/fabric/item/report/definition/page/2.0.0/schema.json",
    "VisualContainer": "https://developer.microsoft.com/json-schemas/fabric/item/report/definition/visualContainer/2.3.0/schema.json",
    "Bookmark": "https://developer.microsoft.com/json-schemas/fabric/item/report/definition/bookmark/1.4.0/schema.json",
    "VersionMetadata": "https://developer.microsoft.com/json-schemas/fabric/item/report/definition/versionMetadata/1.0.0/schema.json",
    "PagesMetadata": "https://developer.microsoft.com/json-schemas/fabric/item/report/definition/pagesMetadata/1.0.0/schema.json"
}

remote_cache = {}

def resolve_type_from_ref(base_url, ref_path, current_schema):
    if ref_path.startswith('#'):
        parts = ref_path.strip('#/').split('/')
        curr = current_schema
        for p in parts: curr = curr.get(p, {})
        return curr.get('type', 'object')
    target_url = urljoin(base_url, ref_path.split('#')[0])
    if target_url not in remote_cache:
        res = requests.get(target_url); res.raise_for_status()
        remote_cache[target_url] = res.json()
    schema = remote_cache[target_url]
    if '#' in ref_path:
        parts = ref_path.split('#')[1].strip('/').split('/')
        for p in parts: schema = schema.get(p, {})
    return schema.get('type', 'object')

def clean_schema(obj, base_url, root_schema):
    if isinstance(obj, dict):
        if "description" in obj and "schema to use for an item" in obj["description"]: return {"type": "string"}
        if "$ref" in obj and isinstance(obj["$ref"], str):
            actual_type = resolve_type_from_ref(base_url, obj["$ref"], root_schema)
            return {"anyOf": [{"type": actual_type}, {"type": "null"}], "default": None}
        return {k: clean_schema(v, base_url, root_schema) for k, v in obj.items()}
    return [clean_schema(i, base_url, root_schema) for i in obj] if isinstance(obj, list) else obj

# IST Configuration (UTC + 5:30)
ist_now = datetime.now(timezone.utc) + timedelta(hours=5, minutes=30)
timestamp = ist_now.strftime("%Y-%m-%d %H:%M:%S IST")

final_code = [
    f"# Generated on: {timestamp}",
    "from __future__ import annotations",
    "from typing import Literal, Any, Union, List, Optional, Dict",
    "from pydantic import BaseModel, ConfigDict, Field, constr, RootModel"
]

for name, url in urls.items():
    schema_obj = requests.get(url).json()
    cleaned = clean_schema(schema_obj, url, schema_obj)
    output = generate(json.dumps(cleaned), input_file_type=InputFileType.JsonSchema, output_model_type="pydantic_v2.BaseModel")
    block = re.sub(r'^(from __future__|from pydantic|from typing).*$', '', output, flags=re.MULTILINE)
    final_code.append(f"# --- {name} ---\n" + block.strip())

with open("/content/Power_BI_Spark_Labs/Power BI/Automations/Power Bi Desktop/Mach3/fabric_models.py", "w") as f:
    f.write("\n".join(final_code))
print(f"fabric_models.py created successfully at {timestamp}.")

fabric_models.py created and schemas cached in /content/Power_BI_Spark_Labs/Power BI/Automations/Power Bi Desktop/Mach3/schemas_cache
